# Train-only WoE, IV and monotonic bins

These cells were executed against the real project outputs. The complete training
and SQL implementation is in the linked source modules. This walkthrough reads and
verifies executed artifacts rather than silently retraining the expensive models.
Home Credit is a consumer-lending proxy; validation is random holdout, never true OOT.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'reports/metrics.json').exists():
    ROOT=ROOT.parent
metrics=json.loads((ROOT/'reports/metrics.json').read_text())


## WoE and IV

In [2]:
display(pd.read_csv(ROOT/'reports/iv_ranking.csv').head(25))
bins=pd.read_csv(ROOT/'reports/woe_bins.csv')
display(bins.head(25))
assert bins.groupby('feature')['count'].sum().eq(metrics['validation']['scorecard']['train']['rows']).all()
print('All bin counts match model-training rows.')

,Unnamed: 0,iv
0,EXT_SOURCE_3,0.316338
1,EXT_SOURCE_2,0.314522
2,EXT_SOURCE_1,0.150310
3,bureau_avg_credit_day,0.112537
4,DAYS_EMPLOYED,0.109552
5,bureau_debt_credit_ratio,0.093654
6,OCCUPATION_TYPE,0.083398
7,age_years,0.082660
8,DAYS_BIRTH,0.082656
9,credit_goods_ratio,0.082160


,bin,count,bad,good,good_dist,bad_dist,woe,iv_component,bad_rate,feature,bin_definition
0,Cash loans,133271,11156,122115,0.899966,0.936184,-0.039455,0.001429,0.083709,NAME_CONTRACT_TYPE,Cash loans
1,Revolving loans,14333,760,13573,0.100034,0.063816,0.449499,0.016280,0.053024,NAME_CONTRACT_TYPE,Revolving loans
2,N,97227,8296,88931,0.655407,0.696190,-0.060367,0.002462,0.085326,FLAG_OWN_CAR,N
3,Y,50377,3620,46757,0.344593,0.303810,0.125963,0.005137,0.071858,FLAG_OWN_CAR,Y
4,N,45201,3766,41435,0.305371,0.316061,-0.034408,0.000368,0.083317,FLAG_OWN_REALTY,N
5,Y,102403,8150,94253,0.694629,0.683939,0.015509,0.000166,0.079588,FLAG_OWN_REALTY,Y
6,0,132773,10624,122149,0.900213,0.891504,0.009722,0.000085,0.080016,CNT_CHILDREN,"(-inf, 1]"
7,1,12792,1107,11685,0.086119,0.092931,-0.076118,0.000518,0.086538,CNT_CHILDREN,"(1, 2]"
8,2,2039,185,1854,0.013667,0.015565,-0.130046,0.000247,0.090731,CNT_CHILDREN,"(2, inf]"
9,0,88564,7530,81034,0.597201,0.631833,-0.056371,0.001952,0.085023,AMT_INCOME_TOTAL,"(-inf, 165150]"


All bin counts match model-training rows.


## Binning method

See src/woe_binning.py. Numeric training quantiles are merged for monotonic event
rates; missing has its own bin. Rare categories are pooled. WoE = ln(smoothed good share /
smoothed bad share); IV sums (good share - bad share) x WoE. Bin edges and selection never
use test/holdout labels. Missing/unseen categories and smoothing are documented.